In [4]:
from ultralytics import YOLO
import numpy as np
if not hasattr(np, "object"):
    np.object = object


# Configure the tracking parameters and run the tracker
model_path = "E:\Soccer\\football\\football_analysis-main\Storage\models\yolo\yolo11n.pt"
model = YOLO(model_path)
results = model.track(source="E:\Soccer\\football\\football_analysis-main\IO\input_videos\\test.mp4", conf=0.3, iou=0.5, show=True, save=True, tracker="botsort.yaml")

C:\Users\x1919\AppData\Local\Temp\ipykernel_37724\3240364968.py:3: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):



WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/750) E:\Soccer\football\football_analysis-main\IO\input_videos\test.mp4: 384x640 21 persons, 56.3ms
video 1/1 (frame 2/750) E:\Soccer\football\football_analysis-main\IO\input_videos\test.mp4: 384x640 21 persons, 43.8ms
video 1/1 (frame 3/750) E:\Soccer\football\football_analysis-main\IO\input_videos\test.mp4: 384x640 19 persons, 43.4ms
video 1/1 (frame 4/750) E:\Soccer\football\football_analysis-main\IO\input_videos\test.mp4: 384x640 18

In [ ]:
"""
BoTSORT 跟踪器测试

本测试展示如何使用项目中的 BoTSORT 跟踪器进行多目标跟踪。
测试包括：
1. 导入必要的模块
2. 初始化 BoTSORT 跟踪器
3. 使用 YOLO 进行目标检测
4. 使用 BoTSORT 进行跟踪
5. 可视化跟踪结果
"""

import os
import cv2
import numpy as np
from ultralytics import YOLO
from Module.common.trackers.bot_sort.bot_sort import BoTSORT
from Module.common.utils import read_video
import matplotlib.pyplot as plt

# ==================== 配置参数 ====================
# 项目根目录
PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))

# 模型路径
YOLO_MODEL_PATH = os.path.join(PROJECT_ROOT, 'Storage', 'models', 'yolo', 'yolo11n.pt')
# 注意：BoTSORT 需要 ReID 模型，如果没有可以使用其他跟踪器
# 这里假设 ReID 模型路径（如果没有，后续会有说明）
REID_MODEL_PATH = None  # 如果没有 ReID 模型，可以使用其他方法

# 视频路径
VIDEO_PATH = os.path.join(PROJECT_ROOT, 'IO', 'input_videos', 'test.mp4')
OUTPUT_PATH = os.path.join(PROJECT_ROOT, 'IO', 'output_videos', 'botsort_test.avi')

# 设备配置
DEVICE = "cuda" if os.getenv('CUDA_VISIBLE_DEVICES') else "cpu"
USE_FP16 = DEVICE == "cuda"  # GPU 时使用半精度

print(f"📌 项目根目录: {PROJECT_ROOT}")
print(f"📌 设备: {DEVICE}")
print(f"📌 视频路径: {VIDEO_PATH}")
print(f"📌 YOLO 模型路径: {YOLO_MODEL_PATH}")
print("\n" + "="*50 + "\n")

# ==================== 初始化模型 ====================
print("🔧 初始化 YOLO 检测模型...")
yolo_model = YOLO(YOLO_MODEL_PATH)
print("✅ YOLO 模型加载完成\n")

# ==================== 初始化 BoTSORT 跟踪器 ====================
# 注意：BoTSORT 需要 ReID 模型权重
# 如果没有 ReID 模型，可以使用默认的跟踪器
print("🔧 初始化 BoTSORT 跟踪器...")
try:
    # 如果有 ReID 模型，使用 BoTSORT
    if REID_MODEL_PATH and os.path.exists(REID_MODEL_PATH):
        tracker = BoTSORT(
            model_weights=REID_MODEL_PATH,
            device=DEVICE,
            fp16=USE_FP16,
            track_high_thresh=0.45,      # 高置信度阈值
            new_track_thresh=0.6,        # 新轨迹阈值
            track_buffer=30,             # 轨迹缓冲帧数
            match_thresh=0.8,            # 匹配阈值
            proximity_thresh=0.5,        # 邻近阈值
            appearance_thresh=0.25,      # 外观阈值
            cmc_method='sparseOptFlow',  # 相机运动补偿方法（速度最快）
            frame_rate=30,               # 视频帧率
            lambda_=0.985                # 运动-外观融合权重
        )
        print("✅ BoTSORT 跟踪器初始化完成（使用 ReID 模型）\n")
    else:
        # 如果没有 ReID 模型，提示用户
        print("⚠️  未找到 ReID 模型，无法使用 BoTSORT")
        print("   提示：BoTSORT 需要 ReID 模型权重才能工作")
        print("   可以：")
        print("   1. 提供 ReID 模型路径")
        print("   2. 使用项目中的通用跟踪器（Tracker）\n")
        raise FileNotFoundError("ReID model not found")
        
except Exception as e:
    print(f"❌ BoTSORT 初始化失败: {e}\n")
    print("💡 使用替代方案：使用 YOLO 内置跟踪器进行测试\n")
    tracker = None

# ==================== 测试函数：单帧跟踪 ====================
def test_single_frame(frame, frame_id=0):
    """
    测试单帧的跟踪功能
    
    Args:
        frame: 视频帧（numpy 数组）
        frame_id: 帧ID
    """
    print(f"📹 处理第 {frame_id} 帧...")
    
    # 1. 使用 YOLO 进行检测
    results = yolo_model(frame, conf=0.3, verbose=False)
    detections = []
    
    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        for i in range(len(boxes)):
            # 提取边界框坐标
            x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy()
            conf = float(boxes.conf[i].cpu().numpy())
            cls = int(boxes.cls[i].cpu().numpy())
            
            # BoTSORT 输入格式: [x1, y1, x2, y2, conf, cls, tracklab_id]
            # tracklab_id 可以设为 None 或检测器的 ID
            detections.append([x1, y1, x2, y2, conf, cls, None])
        
        print(f"   ✅ 检测到 {len(detections)} 个目标")
        
        # 2. 如果有跟踪器，使用跟踪器
        if tracker is not None:
            detections_array = np.array(detections, dtype=np.float32)
            tracking_results = tracker.update(detections_array, frame)
            
            print(f"   ✅ 跟踪结果: {len(tracking_results)} 个轨迹")
            
            # 返回跟踪结果
            return detections, tracking_results
        else:
            return detections, None
    
    return [], None

# ==================== 测试函数：视频跟踪 ====================
def test_video_tracking(video_path, max_frames=50, show_progress=True):
    """
    测试视频跟踪功能
    
    Args:
        video_path: 视频文件路径
        max_frames: 最大处理帧数（用于快速测试）
        show_progress: 是否显示进度
    """
    if tracker is None:
        print("❌ 跟踪器未初始化，无法进行视频跟踪测试")
        return
    
    print(f"🎬 开始处理视频: {video_path}")
    print(f"📊 最大处理帧数: {max_frames}\n")
    
    # 读取视频
    frames = read_video(video_path)
    total_frames = min(len(frames), max_frames)
    
    print(f"📹 视频总帧数: {len(frames)}, 将处理: {total_frames} 帧\n")
    
    # 用于存储统计信息
    stats = {
        'total_detections': 0,
        'total_tracks': 0,
        'unique_track_ids': set()
    }
    
    # 处理前几帧作为示例
    for frame_id in range(total_frames):
        if show_progress and frame_id % 10 == 0:
            print(f"处理进度: {frame_id}/{total_frames} ({frame_id/total_frames*100:.1f}%)")
        
        frame = frames[frame_id]
        detections, tracking_results = test_single_frame(frame, frame_id)
        
        stats['total_detections'] += len(detections)
        
        if tracking_results is not None:
            stats['total_tracks'] += len(tracking_results)
            for result in tracking_results:
                if len(result) >= 5:
                    track_id = result[4]  # track_id 在第5个位置
                    stats['unique_track_ids'].add(track_id)
    
    print("\n" + "="*50)
    print("📊 跟踪统计信息:")
    print(f"   - 总检测数: {stats['total_detections']}")
    print(f"   - 总跟踪数: {stats['total_tracks']}")
    print(f"   - 唯一轨迹ID数: {len(stats['unique_track_ids'])}")
    print("="*50 + "\n")
    
    return stats

# ==================== 执行测试 ====================
if __name__ == "__main__":
    # 测试1: 单帧测试（快速验证）
    print("🧪 测试1: 单帧跟踪测试")
    print("-" * 50)
    
    if os.path.exists(VIDEO_PATH):
        frames = read_video(VIDEO_PATH)
        if len(frames) > 0:
            test_frame = frames[0]
            detections, tracking_results = test_single_frame(test_frame, 0)
            
            if tracking_results is not None and len(tracking_results) > 0:
                print(f"\n✅ 单帧测试成功！")
                print(f"   检测结果: {len(detections)} 个目标")
                print(f"   跟踪结果: {len(tracking_results)} 个轨迹")
                print("\n   前3个跟踪结果示例:")
                for i, result in enumerate(tracking_results[:3]):
                    if len(result) >= 8:
                        x1, y1, x2, y2, track_id, cls, score, tracklab_id = result[:8]
                        print(f"     轨迹 {i+1}: ID={track_id}, 类别={cls}, 置信度={score:.2f}")
            else:
                print("⚠️  未获得跟踪结果")
        else:
            print("❌ 视频文件为空")
    else:
        print(f"❌ 视频文件不存在: {VIDEO_PATH}")
    
    print("\n" + "="*50 + "\n")
    
    # 测试2: 视频跟踪测试（可选，如果 ReID 模型可用）
    if tracker is not None and os.path.exists(VIDEO_PATH):
        print("🧪 测试2: 视频跟踪测试（前50帧）")
        print("-" * 50)
        try:
            stats = test_video_tracking(VIDEO_PATH, max_frames=50)
            print("✅ 视频跟踪测试完成！")
        except Exception as e:
            print(f"❌ 视频跟踪测试失败: {e}")
    else:
        print("⏭️  跳过视频跟踪测试（需要 ReID 模型）")
    
    print("\n" + "="*50)
    print("📝 测试总结")
    print("="*50)
    print("如果测试成功，说明 BoTSORT 跟踪器可以正常工作。")
    print("如果遇到问题，请检查：")
    print("1. ReID 模型路径是否正确")
    print("2. 视频文件是否存在")
    print("3. CUDA 环境是否正确配置（如果使用 GPU）")
    print("="*50)
